# 13｜深層神經網路訓練_最佳化與正則化

在同一個狹長損失面比較一般梯度下降與 Momentum。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
A = np.diag([1.0, 30.0])
def optimize(momentum=0.0, eta=0.035, steps=80):
    theta = np.array([4.0, 1.5]); velocity = np.zeros(2); losses = []
    for _ in range(steps):
        losses.append(0.5 * theta @ A @ theta)
        grad = A @ theta
        velocity = momentum * velocity + grad
        theta -= eta * velocity
    return np.array(losses), theta

plain_loss, plain_theta = optimize(momentum=0.0)
mom_loss, mom_theta = optimize(momentum=0.8)
pd.DataFrame({'method': ['GD', 'Momentum'], 'final_loss': [plain_loss[-1], mom_loss[-1]],
              'theta_0': [plain_theta[0], mom_theta[0]], 'theta_1': [plain_theta[1], mom_theta[1]]})

## 執行結果圖

In [ ]:
plt.semilogy(plain_loss, label='gradient descent')
plt.semilogy(mom_loss, label='momentum=0.8')
plt.xlabel('update'); plt.ylabel('quadratic loss'); plt.title('Momentum crosses the shallow direction faster')
plt.legend()
savefig('13_optimizer_regularization_demo')
report('optimizer_regularization', {'gd_final_loss': plain_loss[-1], 'momentum_final_loss': mom_loss[-1]})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。